In [20]:
import os
EXPORT_PATH = os.path.join(os.getcwd(), 'powerbi_exports')
os.makedirs(EXPORT_PATH, exist_ok=True)
print(f"✅ Export path set: {EXPORT_PATH}")

✅ Export path set: d:\YUVRAJ\YUVRAJ PROJECTS\supply-chain-intelligence\powerbi_exports



# SUPPLY CHAIN INTELLIGENCE
# Notebook 4: Power BI Data Export

In [ ]:


import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
engine = create_engine(
    f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}"
    f"/{os.getenv('DB_NAME')}"
)

# Output folder
os.makedirs('../data/powerbi_exports', exist_ok=True)
print("✅ Setup complete!")

✅ Setup complete!



# CHECK ACTUAL COLUMN NAMES IN ALL TABLES


In [ ]:


tables = ['orders', 'customers', 'delivery_performance', 'customer_complaints']

for table in tables:
    query = f"""
    SELECT column_name, data_type 
    FROM information_schema.columns 
    WHERE table_name = '{table}'
    ORDER BY ordinal_position
    """
    cols = pd.read_sql(query, engine)
    print(f"\n📋 TABLE: {table}")
    print(f"   Columns ({len(cols)}):")
    for _, row in cols.iterrows():
        print(f"   - {row['column_name']} ({row['data_type']})")


📋 TABLE: orders
   Columns (24):
   - order_item_id (integer)
   - order_id (integer)
   - customer_id (integer)
   - order_date (date)
   - order_status (character varying)
   - product_name (character varying)
   - product_price (numeric)
   - category_name (character varying)
   - department_name (character varying)
   - order_quantity (integer)
   - order_item_total (numeric)
   - sales (numeric)
   - order_profit_per_order (numeric)
   - benefit_per_order (numeric)
   - order_item_discount (numeric)
   - order_item_discount_rate (numeric)
   - order_item_profit_ratio (numeric)
   - market (character varying)
   - order_region (character varying)
   - order_country (character varying)
   - order_state (character varying)
   - order_city (character varying)
   - transaction_type (character varying)
   - created_at (timestamp without time zone)

📋 TABLE: customers
   Columns (13):
   - customer_id (integer)
   - customer_fname (character varying)
   - customer_lname (character varyi

In [22]:
query = """
SELECT 
    o.order_item_id,
    o.order_id,
    o.customer_id,
    o.order_date,
    EXTRACT(YEAR FROM o.order_date)  AS order_year,
    EXTRACT(MONTH FROM o.order_date) AS order_month,
    TO_CHAR(o.order_date, 'YYYY-MM') AS order_year_month,
    o.category_name,
    o.department_name,
    o.product_name,
    o.product_price,
    o.market,
    o.order_region,
    o.order_country,
    o.order_state,
    o.order_city,
    o.sales,
    o.order_quantity,
    o.order_item_discount,
    o.order_item_discount_rate,
    o.order_item_profit_ratio,
    o.order_profit_per_order,
    o.benefit_per_order,
    o.order_item_total,
    o.transaction_type,
    o.order_status
FROM orders o
WHERE o.order_date IS NOT NULL
ORDER BY o.order_date
"""

fact_orders = pd.read_sql(query, engine)

fact_orders['order_date'] = pd.to_datetime(
    fact_orders['order_date']
).dt.date

fact_orders.to_csv(
    os.path.join(EXPORT_PATH, 'fact_orders.csv'),
    index=False
)
print(f"✅ fact_orders: {len(fact_orders):,} rows exported")

✅ fact_orders: 180,519 rows exported


In [25]:
fact_delivery = pd.read_sql("""
SELECT 
    d.delivery_id, d.order_item_id, d.order_id, d.customer_id,
    d.shipping_mode, d.days_for_shipping_real,
    d.days_for_shipment_scheduled, d.delivery_delay_days,
    d.late_delivery_risk, d.delivery_status, d.shipping_date,
    CASE
        WHEN d.delivery_delay_days <= 0 THEN 'On Time'
        WHEN d.delivery_delay_days <= 2 THEN 'Slightly Late (1-2d)'
        WHEN d.delivery_delay_days <= 5 THEN 'Moderately Late (3-5d)'
        ELSE 'Severely Late (5d+)'
    END AS delay_category
FROM delivery_performance d
""", engine)

fact_delivery['shipping_date'] = pd.to_datetime(
    fact_delivery['shipping_date']
).dt.date

fact_delivery.to_csv(
    os.path.join(EXPORT_PATH, 'fact_delivery.csv'), index=False
)
print(f"✅ fact_delivery: {len(fact_delivery):,} rows exported")

✅ fact_delivery: 180,519 rows exported


In [26]:
fact_complaints = pd.read_sql("""
SELECT 
    cc.complaint_id, cc.order_id, cc.customer_id,
    cc.complaint_date,
    EXTRACT(YEAR FROM cc.complaint_date)  AS complaint_year,
    EXTRACT(MONTH FROM cc.complaint_date) AS complaint_month,
    cc.complaint_category, cc.resolution_status,
    cc.resolution_days, cc.satisfaction_score, cc.is_churned,
    CASE 
        WHEN cc.satisfaction_score >= 4 THEN 'Satisfied (4-5)'
        WHEN cc.satisfaction_score = 3  THEN 'Neutral (3)'
        ELSE 'Dissatisfied (1-2)'
    END AS satisfaction_tier
FROM customer_complaints cc
WHERE cc.complaint_date IS NOT NULL
""", engine)

fact_complaints['complaint_date'] = pd.to_datetime(
    fact_complaints['complaint_date']
).dt.date

fact_complaints.to_csv(
    os.path.join(EXPORT_PATH, 'fact_complaints.csv'), index=False
)
print(f"✅ fact_complaints: {len(fact_complaints):,} rows exported")

✅ fact_complaints: 87,959 rows exported


In [27]:
dim_customers = pd.read_sql("""
SELECT 
    c.customer_id,
    c.customer_fname, c.customer_lname,
    c.customer_fname || ' ' || c.customer_lname AS customer_name,
    c.customer_segment, c.customer_country, c.customer_city,
    c.customer_state, c.customer_zipcode,
    c.latitude, c.longitude,

    MIN(o.order_date)               AS first_order_date,
    MAX(o.order_date)               AS last_order_date,
    COUNT(DISTINCT o.order_id)      AS lifetime_orders,
    ROUND(SUM(o.sales)::numeric, 2) AS lifetime_revenue,
    MAX(cc.is_churned)              AS is_churned,

    CASE
        WHEN (MAX(o.order_date) - MIN(o.order_date)) <= 90
            THEN '0-3 Months'
        WHEN (MAX(o.order_date) - MIN(o.order_date)) <= 180
            THEN '3-6 Months'
        WHEN (MAX(o.order_date) - MIN(o.order_date)) <= 365
            THEN '6-12 Months'
        WHEN (MAX(o.order_date) - MIN(o.order_date)) <= 730
            THEN '12-24 Months'
        ELSE '24+ Months'
    END AS tenure_bucket,

    ROUND(
        (SUM(o.sales) / NULLIF(COUNT(DISTINCT o.order_id), 0))
        * COUNT(DISTINCT o.order_id) * 1.5
    , 2) AS projected_clv,

    CASE
        WHEN MAX(cc.is_churned) = 1 THEN 'Churned'
        WHEN AVG(cc.satisfaction_score) < 2.5 
            AND SUM(d.late_delivery_risk) > 3 THEN 'HIGH'
        WHEN AVG(cc.satisfaction_score) < 3.5 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS risk_tier

FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN delivery_performance d ON o.order_item_id = d.order_item_id
LEFT JOIN customer_complaints cc ON o.order_id = cc.order_id
GROUP BY 
    c.customer_id, c.customer_fname, c.customer_lname,
    c.customer_segment, c.customer_country, c.customer_city,
    c.customer_state, c.customer_zipcode,
    c.latitude, c.longitude
""", engine)

dim_customers['first_order_date'] = pd.to_datetime(
    dim_customers['first_order_date']
).dt.date
dim_customers['last_order_date'] = pd.to_datetime(
    dim_customers['last_order_date']
).dt.date

dim_customers.to_csv(
    os.path.join(EXPORT_PATH, 'dim_customers.csv'), index=False
)
print(f"✅ dim_customers: {len(dim_customers):,} customers exported")
print(f"\n   Risk Tier Distribution:")
print(dim_customers['risk_tier'].value_counts())
print(f"\n   Tenure Distribution:")
print(dim_customers['tenure_bucket'].value_counts())

✅ dim_customers: 20,652 customers exported

   Risk Tier Distribution:
risk_tier
Churned    8967
LOW        6036
MEDIUM     5302
HIGH        347
Name: count, dtype: int64

   Tenure Distribution:
tenure_bucket
0-3 Months      9166
12-24 Months    5350
24+ Months      4505
6-12 Months     1240
3-6 Months       391
Name: count, dtype: int64


In [28]:
# TABLE 5: agg_monthly_kpis
agg_monthly = pd.read_sql("""
SELECT
    TO_CHAR(o.order_date, 'YYYY-MM')    AS year_month,
    EXTRACT(YEAR FROM o.order_date)     AS year,
    EXTRACT(MONTH FROM o.order_date)    AS month,
    o.market, o.order_region,

    COUNT(DISTINCT o.order_id)               AS total_orders,
    COUNT(DISTINCT o.customer_id)            AS active_customers,
    ROUND(SUM(o.sales)::numeric, 2)          AS total_revenue,
    ROUND(SUM(o.order_profit_per_order)::numeric, 2) AS total_profit,
    ROUND(AVG(o.order_profit_per_order)::numeric, 2) AS avg_profit_per_order,
    ROUND(AVG(o.order_item_discount_rate)::numeric, 4) AS avg_discount_rate,

    SUM(d.late_delivery_risk)                AS late_deliveries,
    COUNT(d.delivery_id)                     AS total_deliveries,
    ROUND(
        SUM(d.late_delivery_risk)::numeric /
        NULLIF(COUNT(d.delivery_id), 0) * 100
    , 2)                                     AS late_delivery_pct,

    COUNT(cc.complaint_id)                   AS total_complaints,
    ROUND(AVG(cc.satisfaction_score)::numeric, 2) AS avg_satisfaction,
    SUM(cc.is_churned)                       AS churned_customers

FROM orders o
JOIN delivery_performance d ON o.order_item_id = d.order_item_id
LEFT JOIN customer_complaints cc ON o.order_id = cc.order_id
WHERE o.order_date IS NOT NULL
GROUP BY
    TO_CHAR(o.order_date, 'YYYY-MM'),
    EXTRACT(YEAR FROM o.order_date),
    EXTRACT(MONTH FROM o.order_date),
    o.market, o.order_region
ORDER BY year_month
""", engine)

agg_monthly.to_csv(
    os.path.join(EXPORT_PATH, 'agg_monthly_kpis.csv'), index=False
)
print(f"✅ agg_monthly_kpis: {len(agg_monthly):,} rows exported")

# TABLE 6: agg_customer_risk_scores
agg_risk = pd.read_sql("""
SELECT
    c.customer_id,
    c.customer_fname || ' ' || c.customer_lname AS customer_name,
    c.customer_segment, c.customer_country,

    COUNT(DISTINCT o.order_id)       AS total_orders,
    ROUND(SUM(o.sales)::numeric, 2)  AS lifetime_revenue,
    ROUND(
        (SUM(o.sales) / NULLIF(COUNT(DISTINCT o.order_id), 0))
        * COUNT(DISTINCT o.order_id) * 1.5
    , 2)                             AS projected_clv,

    ROUND(AVG(cc.satisfaction_score)::numeric, 2) AS avg_satisfaction,
    COUNT(cc.complaint_id)           AS total_complaints,
    SUM(CASE WHEN cc.resolution_status = 'Unresolved'
        THEN 1 ELSE 0 END)           AS unresolved_complaints,

    ROUND(
        SUM(d.late_delivery_risk)::numeric /
        NULLIF(COUNT(d.delivery_id), 0) * 100
    , 1)                             AS late_delivery_pct,

    MAX(cc.is_churned)               AS is_churned,

    ROUND((
        (1 - COALESCE(AVG(cc.satisfaction_score), 3) / 5.0) * 40
        + (SUM(d.late_delivery_risk)::float /
           NULLIF(COUNT(d.delivery_id), 0)) * 35
        + (COUNT(CASE WHEN cc.resolution_status = 'Unresolved'
                 THEN 1 END)::float /
           NULLIF(COUNT(cc.complaint_id), 0)) * 25
    )::numeric, 1)                   AS risk_score

FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN delivery_performance d ON o.order_item_id = d.order_item_id
LEFT JOIN customer_complaints cc ON o.order_id = cc.order_id
GROUP BY
    c.customer_id, c.customer_fname,
    c.customer_lname, c.customer_segment,
    c.customer_country
ORDER BY risk_score DESC
""", engine)

agg_risk['risk_tier'] = pd.cut(
    agg_risk['risk_score'],
    bins=[-1, 30, 60, 100],
    labels=['LOW', 'MEDIUM', 'HIGH']
)

agg_risk.to_csv(
    os.path.join(EXPORT_PATH, 'agg_customer_risk_scores.csv'), index=False
)
print(f"✅ agg_customer_risk_scores: {len(agg_risk):,} rows exported")
print(f"\n   Risk Tier Distribution:")
print(agg_risk['risk_tier'].value_counts())

✅ agg_monthly_kpis: 210 rows exported
✅ agg_customer_risk_scores: 20,652 rows exported

   Risk Tier Distribution:
risk_tier
MEDIUM    7904
HIGH      6943
LOW       1026
Name: count, dtype: int64


In [29]:
orders_raw = pd.read_sql("""
SELECT 
    o.customer_id, o.order_id, o.order_date,
    MIN(o.order_date) OVER (
        PARTITION BY o.customer_id
    ) AS first_order_date
FROM orders o
WHERE o.order_date IS NOT NULL
""", engine)

orders_raw['order_date']       = pd.to_datetime(orders_raw['order_date'])
orders_raw['first_order_date'] = pd.to_datetime(orders_raw['first_order_date'])

orders_raw['cohort_month']       = orders_raw['first_order_date'].dt.to_period('M')
orders_raw['order_month']        = orders_raw['order_date'].dt.to_period('M')
orders_raw['months_since_first'] = (
    orders_raw['order_month'] - orders_raw['cohort_month']
).apply(lambda x: x.n)

cohort_data = orders_raw.groupby(
    ['cohort_month', 'months_since_first']
)['customer_id'].nunique().reset_index()
cohort_data.columns = ['cohort_month', 'months_since_first', 'customers']

cohort_size = cohort_data[
    cohort_data['months_since_first'] == 0
][['cohort_month', 'customers']].rename(columns={'customers': 'cohort_size'})

cohort_final = cohort_data.merge(cohort_size, on='cohort_month')
cohort_final['retention_rate'] = (
    cohort_final['customers'] / cohort_final['cohort_size'] * 100
).round(1)
cohort_final['cohort_month'] = cohort_final['cohort_month'].astype(str)
cohort_final = cohort_final[cohort_final['months_since_first'] <= 12]

cohort_final.to_csv(
    os.path.join(EXPORT_PATH, 'agg_cohort_matrix.csv'), index=False
)
print(f"✅ agg_cohort_matrix: {len(cohort_final):,} rows exported")
print(cohort_final.head(5))

✅ agg_cohort_matrix: 355 rows exported
  cohort_month  months_since_first  customers  cohort_size  retention_rate
0      2015-01                   0       1668         1668           100.0
1      2015-01                   1        191         1668            11.5
2      2015-01                   2        226         1668            13.5
3      2015-01                   3        215         1668            12.9
4      2015-01                   4        208         1668            12.5


In [30]:
print("📁 Power BI Export Folder — Files Ready:")
print("=" * 55)
total_size = 0
for f in sorted(os.listdir(EXPORT_PATH)):
    path = os.path.join(EXPORT_PATH, f)
    size_kb = os.path.getsize(path) / 1024
    total_size += size_kb
    df_check = pd.read_csv(path)
    print(f"  ✅ {f}")
    print(f"      Rows: {len(df_check):,} | "
          f"Cols: {len(df_check.columns)} | "
          f"Size: {size_kb:.1f} KB")

print(f"\n📊 Total export size: {total_size/1024:.2f} MB")
print(f"\n📍 Folder location:")
print(f"   {EXPORT_PATH}")
print(f"\n🚀 Ready to import into Power BI!")

📁 Power BI Export Folder — Files Ready:
  ✅ agg_cohort_matrix.csv
      Rows: 355 | Cols: 5 | Size: 8.1 KB
  ✅ agg_customer_risk_scores.csv
      Rows: 20,652 | Cols: 14 | Size: 1650.5 KB
  ✅ agg_monthly_kpis.csv
      Rows: 210 | Cols: 17 | Size: 22.4 KB
  ✅ dim_customers.csv
      Rows: 20,652 | Cols: 19 | Size: 3042.0 KB
  ✅ fact_complaints.csv
      Rows: 87,959 | Cols: 12 | Size: 7731.9 KB
  ✅ fact_delivery.csv
      Rows: 180,519 | Cols: 12 | Size: 15659.7 KB
  ✅ fact_orders.csv
      Rows: 180,519 | Cols: 26 | Size: 39801.6 KB

📊 Total export size: 66.32 MB

📍 Folder location:
   d:\YUVRAJ\YUVRAJ PROJECTS\supply-chain-intelligence\powerbi_exports

🚀 Ready to import into Power BI!


In [1]:
import pandas as pd
import os

# Cohort matrix reload karo
cohort_df = pd.read_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv')
)

print(cohort_df.head(20))
print("\nColumns:", cohort_df.columns.tolist())
print("\nRetention rate range:")
print(cohort_df['retention_rate'].describe())

   cohort_month  months_since_first  customers  cohort_size  retention_rate
0       2015-01                   0       1668         1668           100.0
1       2015-01                   1        191         1668            11.5
2       2015-01                   2        226         1668            13.5
3       2015-01                   3        215         1668            12.9
4       2015-01                   4        208         1668            12.5
5       2015-01                   5        233         1668            14.0
6       2015-01                   6        196         1668            11.8
7       2015-01                   7        203         1668            12.2
8       2015-01                   8        224         1668            13.4
9       2015-01                   9        224         1668            13.4
10      2015-01                  10        207         1668            12.4
11      2015-01                  11        227         1668            13.6
12      2015

In [2]:
import pandas as pd
import os

cohort_df = pd.read_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv')
)

# cohort_month ko text format mein convert karo
# "2015-01" → "Jan 2015" 
cohort_df['cohort_month'] = pd.to_datetime(
    cohort_df['cohort_month']
).dt.strftime('%b %Y')

# Re-export karo
cohort_df.to_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv'),
    index=False
)

print("✅ Done!")
print(cohort_df['cohort_month'].unique())
print(cohort_df.head())

✅ Done!
<ArrowStringArray>
['Jan 2015', 'Feb 2015', 'Mar 2015', 'Apr 2015', 'May 2015', 'Jun 2015',
 'Jul 2015', 'Aug 2015', 'Sep 2015', 'Oct 2015', 'Nov 2015', 'Dec 2015',
 'Jan 2016', 'Feb 2016', 'Mar 2016', 'Apr 2016', 'May 2016', 'Jun 2016',
 'Jul 2016', 'Aug 2016', 'Sep 2016', 'Oct 2016', 'Nov 2016', 'Dec 2016',
 'Jan 2017', 'Feb 2017', 'Mar 2017', 'Apr 2017', 'May 2017', 'Jun 2017',
 'Jul 2017', 'Aug 2017', 'Sep 2017', 'Oct 2017', 'Nov 2017', 'Dec 2017',
 'Jan 2018']
Length: 37, dtype: str
  cohort_month  months_since_first  customers  cohort_size  retention_rate
0     Jan 2015                   0       1668         1668           100.0
1     Jan 2015                   1        191         1668            11.5
2     Jan 2015                   2        226         1668            13.5
3     Jan 2015                   3        215         1668            12.9
4     Jan 2015                   4        208         1668            12.5


In [3]:
import pandas as pd
import os

cohort_df = pd.read_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv')
)

print("Before fix:")
print(cohort_df.head(3))

# Fix 1 — cohort_month text mein convert karo
cohort_df['cohort_month'] = pd.to_datetime(
    cohort_df['cohort_month']
).dt.strftime('%b-%Y')

# Fix 2 — retention_rate 10x divide karo
cohort_df['retention_rate'] = (
    cohort_df['retention_rate'] / 10
).round(1)

print("\nAfter fix:")
print(cohort_df.head(5))
print("\nRetention rate range:")
print(cohort_df['retention_rate'].describe())

# Save karo
cohort_df.to_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv'),
    index=False
)
print("\n✅ File saved!")

Before fix:
  cohort_month  months_since_first  customers  cohort_size  retention_rate
0     Jan 2015                   0       1668         1668           100.0
1     Jan 2015                   1        191         1668            11.5
2     Jan 2015                   2        226         1668            13.5

After fix:
  cohort_month  months_since_first  customers  cohort_size  retention_rate
0     Jan-2015                   0       1668         1668            10.0
1     Jan-2015                   1        191         1668             1.2
2     Jan-2015                   2        226         1668             1.4
3     Jan-2015                   3        215         1668             1.3
4     Jan-2015                   4        208         1668             1.2

Retention rate range:
count    355.000000
mean       2.194648
std        2.680585
min        0.200000
25%        1.200000
50%        1.300000
75%        1.500000
max       10.000000
Name: retention_rate, dtype: float64

✅ Fil

C:\Users\Acer\AppData\Local\Temp\ipykernel_8568\3144887969.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cohort_df['cohort_month'] = pd.to_datetime(


In [4]:
import pandas as pd
import os

cohort_df = pd.read_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv')
)

# Retention rate wapas sahi karo (10x multiply)
cohort_df['retention_rate'] = (
    cohort_df['retention_rate'] * 10
).round(1)

# cohort_month format theek hai — Jan-2015 good hai
print("Fixed data:")
print(cohort_df.head(5))
print("\nRetention rate range:")
print(cohort_df['retention_rate'].describe())

cohort_df.to_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv'),
    index=False
)
print("✅ Saved!")

Fixed data:
  cohort_month  months_since_first  customers  cohort_size  retention_rate
0     Jan-2015                   0       1668         1668           100.0
1     Jan-2015                   1        191         1668            12.0
2     Jan-2015                   2        226         1668            14.0
3     Jan-2015                   3        215         1668            13.0
4     Jan-2015                   4        208         1668            12.0

Retention rate range:
count    355.000000
mean      21.946479
std       26.805851
min        2.000000
25%       12.000000
50%       13.000000
75%       15.000000
max      100.000000
Name: retention_rate, dtype: float64
✅ Saved!


In [6]:
import pandas as pd
import os

# Original data se fresh start karo
orders_raw = pd.read_csv(
    os.path.join('powerbi_exports', 'fact_orders.csv')
)

orders_raw['order_date'] = pd.to_datetime(orders_raw['order_date'])
orders_raw['first_order_date'] = orders_raw.groupby(
    'customer_id')['order_date'].transform('min')

orders_raw['cohort_month'] = orders_raw[
    'first_order_date'
].dt.to_period('M')
orders_raw['order_month'] = orders_raw[
    'order_date'
].dt.to_period('M')
orders_raw['months_since_first'] = (
    orders_raw['order_month'] - orders_raw['cohort_month']
).apply(lambda x: x.n)

cohort_data = orders_raw.groupby(
    ['cohort_month', 'months_since_first']
)['customer_id'].nunique().reset_index()
cohort_data.columns = [
    'cohort_month', 'months_since_first', 'customers'
]

cohort_size = cohort_data[
    cohort_data['months_since_first'] == 0
][['cohort_month', 'customers']].rename(
    columns={'customers': 'cohort_size'}
)

cohort_final = cohort_data.merge(cohort_size, on='cohort_month')
cohort_final['retention_pct'] = (
    cohort_final['customers'] / 
    cohort_final['cohort_size'] * 100
).round(1)

# cohort_month string mein convert karo
cohort_final['cohort_label'] = cohort_final[
    'cohort_month'
].astype(str)

# Sirf 0-12 months rakho
cohort_final = cohort_final[
    cohort_final['months_since_first'] <= 12
]

# Sirf yeh columns export karo
export_df = cohort_final[[
    'cohort_label',
    'months_since_first', 
    'customers',
    'cohort_size',
    'retention_pct'
]].copy()

export_df.to_csv(
    os.path.join('powerbi_exports', 'agg_cohort_matrix.csv'),
    index=False
)

print("✅ Exported!")
print(export_df.head(10))
print(f"\nColumns: {export_df.columns.tolist()}")
print(f"\nRetention range:")
print(export_df['retention_pct'].describe())

✅ Exported!
  cohort_label  months_since_first  customers  cohort_size  retention_pct
0      2015-01                   0       1668         1668          100.0
1      2015-01                   1        191         1668           11.5
2      2015-01                   2        226         1668           13.5
3      2015-01                   3        215         1668           12.9
4      2015-01                   4        208         1668           12.5
5      2015-01                   5        233         1668           14.0
6      2015-01                   6        196         1668           11.8
7      2015-01                   7        203         1668           12.2
8      2015-01                   8        224         1668           13.4
9      2015-01                   9        224         1668           13.4

Columns: ['cohort_label', 'months_since_first', 'customers', 'cohort_size', 'retention_pct']

Retention range:
count    355.000000
mean      21.941127
std       26.807564
mi